# Data Cleaning and Integration

The cleaning process covers:

- Six FSA electricity-consumption datasets: L4T, M5R, M5S, M6G, M9R, and M9W
- Two Environment and Climate Change Canada weather stations:
  - Toronto City (6158355)
  - Toronto INTL A (6158731)
- Ontario hourly calendar data

In [77]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [78]:
# Project paths

DATA_PATH = Path("../data/processed")

CONSUMPTION_FILES = {
    "L4T": "hourly_consumption_fsa_l4t.csv",
    "M5R": "hourly_consumption_fsa_m5r.csv",
    "M5S": "hourly_consumption_fsa_m5s.csv",
    "M6G": "hourly_consumption_fsa_m6g.csv",
    "M9R": "hourly_consumption_fsa_m9r.csv",
    "M9W": "hourly_consumption_fsa_m9w.csv",
}

WEATHER_FILES = {
    "TORONTO_CITY": "weather_toronto_city_6158355.csv",
    "TORONTO_INTL_A": "weather_toronto_intl_a_6158731.csv",
}

CALENDAR_FILE = "calendar_hourly_ontario_timestamp.csv"

In [79]:
# Project paths

DATA_PATH = Path("../data/processed")

CONSUMPTION_FILES = {
    "L4T": "hourly_consumption_fsa_l4t.csv",
    "M5R": "hourly_consumption_fsa_m5r.csv",
    "M5S": "hourly_consumption_fsa_m5s.csv",
    "M6G": "hourly_consumption_fsa_m6g.csv",
    "M9R": "hourly_consumption_fsa_m9r.csv",
    "M9W": "hourly_consumption_fsa_m9w.csv",
}

WEATHER_FILES = {
    "TORONTO_CITY": "weather_toronto_city_6158355.csv",
    "TORONTO_INTL_A": "weather_toronto_intl_a_6158731.csv",
}

CALENDAR_FILE = "calendar_hourly_ontario_timestamp.csv"

In [80]:
# Validate expected input files

expected_files = (
    list(CONSUMPTION_FILES.values())
    + list(WEATHER_FILES.values())
    + [CALENDAR_FILE]
)

missing_files = [
    file_name
    for file_name in expected_files
    if not (DATA_PATH / file_name).exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing expected files: {missing_files}"
    )

print(f"All {len(expected_files)} expected files are available.")


All 9 expected files are available.


In [81]:
# Load consumption datasets

consumption_data = {
    fsa: pd.read_csv(DATA_PATH / file_name)
    for fsa, file_name in CONSUMPTION_FILES.items()
}

# Load weather datasets

weather_data = {
    station: pd.read_csv(DATA_PATH / file_name)
    for station, file_name in WEATHER_FILES.items()
}

# Load calendar dataset

calendar = pd.read_csv(DATA_PATH / CALENDAR_FILE)

print("Consumption datasets loaded:", len(consumption_data))
print("Weather datasets loaded:", len(weather_data))
print("Calendar dataset loaded:", calendar.shape)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_40584\2694488215.py:11: DtypeWarning: Columns (13,15,23,30) have mixed types. Specify dtype option on import or set low_memory=False.
  station: pd.read_csv(DATA_PATH / file_name)


Consumption datasets loaded: 6
Weather datasets loaded: 2
Calendar dataset loaded: (52584, 45)


In [82]:
# Validate schema consistency across consumption datasets

reference_fsa = next(iter(consumption_data))
reference_columns = list(consumption_data[reference_fsa].columns)

print(f"Reference dataset: {reference_fsa}")
print(f"Number of columns: {len(reference_columns)}")

for fsa, df in consumption_data.items():
    same_columns = list(df.columns) == reference_columns
    
    print(
        f"{fsa}: "
        f"rows={len(df):,}, "
        f"columns={len(df.columns)}, "
        f"same_schema={same_columns}"
    )

Reference dataset: L4T
Number of columns: 11
L4T: rows=215,697, columns=11, same_schema=True
M5R: rows=101,596, columns=11, same_schema=True
M5S: rows=103,032, columns=11, same_schema=True
M6G: rows=203,915, columns=11, same_schema=True
M9R: rows=209,729, columns=11, same_schema=True
M9W: rows=212,492, columns=11, same_schema=True


In [83]:
# Identify column differences across consumption datasets

reference_set = set(reference_columns)

for fsa, df in consumption_data.items():
    current_set = set(df.columns)

    missing = reference_set - current_set
    extra = current_set - reference_set

    print(f"\n{fsa}")
    print("Missing columns:", sorted(missing))
    print("Extra columns:", sorted(extra))


L4T
Missing columns: []
Extra columns: []

M5R
Missing columns: []
Extra columns: []

M5S
Missing columns: []
Extra columns: []

M6G
Missing columns: []
Extra columns: []

M9R
Missing columns: []
Extra columns: []

M9W
Missing columns: []
Extra columns: []


In [84]:
# Validate schema consistency across weather datasets

reference_station = next(iter(weather_data))
reference_weather_columns = list(
    weather_data[reference_station].columns
)

print(f"Reference station: {reference_station}")
print(f"Number of columns: {len(reference_weather_columns)}")

for station, df in weather_data.items():
    same_columns = (
        list(df.columns) == reference_weather_columns
    )

    print(
        f"{station}: "
        f"rows={len(df):,}, "
        f"columns={len(df.columns)}, "
        f"same_schema={same_columns}"
    )

Reference station: TORONTO_CITY
Number of columns: 33
TORONTO_CITY: rows=52,584, columns=33, same_schema=True
TORONTO_INTL_A: rows=52,584, columns=33, same_schema=True


In [85]:
# Identify column differences across weather datasets

reference_weather_set = set(reference_weather_columns)

for station, df in weather_data.items():
    current_set = set(df.columns)

    missing = reference_weather_set - current_set
    extra = current_set - reference_weather_set

    print(f"\n{station}")
    print("Missing columns:", sorted(missing))
    print("Extra columns:", sorted(extra))


TORONTO_CITY
Missing columns: []
Extra columns: []

TORONTO_INTL_A
Missing columns: []
Extra columns: []


In [86]:
# Review calendar dataset structure

print("Calendar dataset")
print("Rows:", f"{len(calendar):,}")
print("Columns:", len(calendar.columns))

print("\nColumn names:")
for column in calendar.columns:
    print("-", column)

Calendar dataset
Rows: 52,584
Columns: 45

Column names:
- timestamp_utc
- timestamp_local
- timestamp_toronto
- date
- year
- quarter
- month
- month_name
- week_of_year
- day_of_year
- day_of_month
- hour
- hour_group
- weekday
- weekday_name
- is_weekend
- is_workday
- is_monday
- is_friday
- season
- is_business_hour
- is_peak_hour_window
- is_month_start
- is_month_end
- is_year_start
- is_year_end
- is_public_holiday
- holiday_name
- is_day_before_holiday
- is_day_after_holiday
- days_to_holiday
- days_after_holiday
- is_long_weekend
- is_daylight_saving_time
- is_dst_transition_day
- is_spring_forward_day
- is_fall_back_day
- utc_offset_hours
- date_index
- hour_sin
- hour_cos
- weekday_sin
- weekday_cos
- month_sin
- month_cos


### Schema Validation Summary

- All six electricity-consumption datasets contain the same 11 columns with a consistent column structure.
- Both weather-station datasets contain the same 33 columns with a consistent column structure.
- The calendar dataset contains 45 calendar and temporal features and maintains its own independent schema.
- No datasets have been concatenated or merged at this stage.
- Structural validation confirms that datasets within each data family are compatible for subsequent quality assessment and cleaning.

## Data Quality Profiling

The datasets are profiled before applying any transformation or cleaning rule.

The assessment focuses on data completeness, duplicate records, data types, key-field validity, and consistency across datasets.

Consumption, weather, and calendar data are evaluated separately because each data family has different structures and data-quality requirements.

### Electricity Consumption Data Quality

The six FSA consumption datasets are assessed using the same quality criteria to identify potential issues before concatenation or transformation.

The assessment includes dataset dimensions, missing values, exact duplicates, data types, categorical consistency, and basic validity checks for the main numeric variables.

In [87]:
# General quality profile for consumption datasets

consumption_profile = []

for fsa, df in consumption_data.items():
    consumption_profile.append({
        "FSA": fsa,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing_Values": int(df.isna().sum().sum()),
        "Duplicate_Rows": int(df.duplicated().sum()),
        "Min_Consumption": df["TOTAL_CONSUMPTION"].min(),
        "Max_Consumption": df["TOTAL_CONSUMPTION"].max(),
        "Min_Premise_Count": df["PREMISE_COUNT"].min(),
        "Max_Premise_Count": df["PREMISE_COUNT"].max()
    })

consumption_profile = pd.DataFrame(consumption_profile)

consumption_profile

,FSA,Rows,Columns,Missing_Values,Duplicate_Rows,Min_Consumption,Max_Consumption,Min_Premise_Count,Max_Premise_Count
0,L4T,215697,11,0,0,38.1,22747.4,51,7443
1,M5R,101596,11,0,0,17.6,15148.0,50,8987
2,M5S,103032,11,0,0,21.3,6793.5,71,5626
3,M6G,203915,11,0,0,21.7,19932.2,43,10555
4,M9R,209729,11,0,0,4.6,14213.9,18,5625
5,M9W,212492,11,0,0,21.3,21569.2,45,8601


In [88]:
# Validate temporal coverage across consumption datasets

temporal_coverage = []

for fsa, df in consumption_data.items():
    timestamps = pd.to_datetime(df["TIMESTAMP"], errors="coerce")

    temporal_coverage.append({
        "FSA": fsa,
        "Start": timestamps.min(),
        "End": timestamps.max(),
        "Invalid_Timestamps": timestamps.isna().sum(),
        "Unique_Timestamps": timestamps.nunique()
    })

temporal_coverage = pd.DataFrame(temporal_coverage)

temporal_coverage

,FSA,Start,End,Invalid_Timestamps,Unique_Timestamps
0,L4T,2021-01-01,2025-12-31 23:00:00,0,43824
1,M5R,2021-01-01,2025-12-31 23:00:00,0,43824
2,M5S,2021-01-01,2025-12-31 23:00:00,0,43824
3,M6G,2021-01-01,2025-12-31 23:00:00,0,43824
4,M9R,2021-01-01,2025-12-31 23:00:00,0,43824
5,M9W,2021-01-01,2025-12-31 23:00:00,0,43824


In [89]:
# Validate temporal coverage of weather and calendar datasets

for station, df in weather_data.items():
    timestamps = pd.to_datetime(
        df["Date/Time (LST)"],
        errors="coerce"
    )

    print(
        station,
        "| Start:", timestamps.min(),
        "| End:", timestamps.max(),
        "| Unique:", timestamps.nunique(),
        "| Invalid:", timestamps.isna().sum()
    )

print("\nCalendar:")

calendar_timestamp = pd.to_datetime(
    calendar["timestamp_local"],
    errors="coerce"
)

print(
    "Start:", calendar_timestamp.min(),
    "| End:", calendar_timestamp.max(),
    "| Unique:", calendar_timestamp.nunique(),
    "| Invalid:", calendar_timestamp.isna().sum()
)

TORONTO_CITY | Start: 2021-01-01 00:00:00 | End: 2026-12-31 23:00:00 | Unique: 52584 | Invalid: 0
TORONTO_INTL_A | Start: 2021-01-01 00:00:00 | End: 2026-12-31 23:00:00 | Unique: 52584 | Invalid: 0

Calendar:
Start: 2021-01-01 00:00:00 | End: 2026-12-31 23:00:00 | Unique: 52584 | Invalid: 0


### Temporal Coverage Validation

All six consumption datasets provide complete and consistent hourly coverage from January 1, 2021 through December 31, 2025, with 43,824 unique timestamps and no invalid timestamps.

Both weather datasets and the calendar dataset cover January 1, 2021 through December 31, 2026, with 52,584 unique hourly timestamps and no invalid timestamps.

The additional 8,760 observations correspond to the 2026 calendar year. These records are retained because they are valid source data. The common historical period available across consumption, weather, and calendar data is 2021–2025 and will be used when constructing the historical modeling dataset.

In [90]:
# Validate customer type and price plan combinations across FSAs

category_profile = []

for fsa, df in consumption_data.items():
    combinations = (
        df.groupby(
            ["CUSTOMER_TYPE", "PRICE_PLAN"],
            dropna=False
        )
        .size()
        .reset_index(name="Rows")
    )

    combinations.insert(0, "FSA", fsa)
    category_profile.append(combinations)

category_profile = pd.concat(
    category_profile,
    ignore_index=True
)

category_profile

,FSA,CUSTOMER_TYPE,PRICE_PLAN,Rows
0,L4T,Residential,Retailer,41866
1,L4T,Residential,TOU,43824
2,L4T,Residential,Tiered,43823
3,L4T,SGS <50kW,Retailer,42360
4,L4T,SGS <50kW,TOU,43824
5,M5R,Residential,TOU,43824
6,M5R,Residential,Tiered,43824
7,M5R,Residential,ULO,13204
8,M5R,SGS <50kW,TOU,744
9,M5S,Residential,TOU,43824


#### Consumption Category Validation Summary

All six FSA datasets provide the same consistent hourly timestamp coverage, but the number of records differs because customer-type and price-plan combinations are not uniformly available across FSAs or across the full study period.

Residential TOU and Tiered records provide broad coverage, while Retailer, ULO, and some SGS <50kW combinations are available only for specific FSAs or portions of the study period.

These differences are therefore related to source-data segmentation rather than missing hourly timestamps at the FSA level.

The customer-type and price-plan combinations represent components of the total hourly electricity consumption for each FSA. Therefore, `TOTAL_CONSUMPTION` will subsequently be aggregated across all available `CUSTOMER_TYPE × PRICE_PLAN` combinations for each `FSA + TIMESTAMP` to construct one hourly electricity-demand observation per FSA.

No category combinations are artificially created or imputed when they are not present in the source data.

### Weather Data Quality Assessment

Weather data from Toronto City (Climate ID 6158355) and Toronto INTL A (Climate ID 6158731) are assessed separately before integration with electricity consumption data.

Both stations share the same 33-column schema and hourly coverage from 2021 through 2026. The assessment focuses on missing values, duplicate records, data types, and the availability of meteorological variables relevant to the forecasting pipeline.

Weather-station assignment to the six FSAs has been defined separately based on geographic proximity and will be applied during data integration.

In [91]:
# General quality profile for weather datasets

weather_profile = []

for station, df in weather_data.items():
    weather_profile.append({
        "Station": station,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing_Values": int(df.isna().sum().sum()),
        "Duplicate_Rows": int(df.duplicated().sum())
    })

weather_profile = pd.DataFrame(weather_profile)

weather_profile

,Station,Rows,Columns,Missing_Values,Duplicate_Rows
0,TORONTO_CITY,52584,33,904303,0
1,TORONTO_INTL_A,52584,33,776593,0


In [92]:
# Missing values by column for each weather station

weather_missing_summary = []

for station, df in weather_data.items():
    for column in df.columns:
        missing_count = int(df[column].isna().sum())
        missing_pct = (missing_count / len(df)) * 100

        weather_missing_summary.append({
            "Station": station,
            "Column": column,
            "Missing_Count": missing_count,
            "Missing_Percent": round(missing_pct, 2)
        })

weather_missing_summary = pd.DataFrame(weather_missing_summary)

weather_missing_summary[
    weather_missing_summary["Missing_Count"] > 0
].sort_values(
    ["Station", "Missing_Percent"],
    ascending=[True, False]
)

,Station,Column,Missing_Count,Missing_Percent
9,TORONTO_CITY,Flag,52584,100.00
11,TORONTO_CITY,Temp Flag,52584,100.00
13,TORONTO_CITY,Dew Point Temp Flag,52584,100.00
15,TORONTO_CITY,Rel Hum Flag,52584,100.00
17,TORONTO_CITY,Precip. Amount Flag,52584,100.00
18,TORONTO_CITY,Wind Dir (10s deg),52584,100.00
19,TORONTO_CITY,Wind Dir Flag,52584,100.00
20,TORONTO_CITY,Wind Spd (km/h),52584,100.00
21,TORONTO_CITY,Wind Spd Flag,52584,100.00
22,TORONTO_CITY,Visibility (km),52584,100.00


#### Missing Weather Observation Pattern

Missing values in the main meteorological variables are examined at the timestamp level to determine whether they represent isolated observations or extended periods without weather measurements.

This distinction is required before selecting any imputation strategy.

In [93]:
# Examine timestamps with missing core weather observations

core_weather_columns = [
    "Temp (°C)",
    "Dew Point Temp (°C)",
    "Rel Hum (%)",
    "Stn Press (kPa)"
]

for station, df in weather_data.items():

    missing_mask = df[core_weather_columns].isna().any(axis=1)

    missing_times = pd.to_datetime(
        df.loc[missing_mask, "Date/Time (LST)"],
        errors="coerce"
    )

    print("\n" + "=" * 60)
    print("Station:", station)
    print("=" * 60)

    print("Rows with missing core weather data:", missing_mask.sum())

    if len(missing_times) > 0:
        print("First missing timestamp:", missing_times.min())
        print("Last missing timestamp:", missing_times.max())

        print("\nMissing observations by year:")
        print(
            missing_times.dt.year
            .value_counts()
            .sort_index()
        )


Station: TORONTO_CITY
Rows with missing core weather data: 3790
First missing timestamp: 2021-07-20 09:00:00
Last missing timestamp: 2026-12-31 23:00:00

Missing observations by year:
Date/Time (LST)
2021       7
2022       5
2023       6
2024       7
2025      16
2026    3749
Name: count, dtype: int64

Station: TORONTO_INTL_A
Rows with missing core weather data: 3799
First missing timestamp: 2021-07-20 19:00:00
Last missing timestamp: 2026-12-31 23:00:00

Missing observations by year:
Date/Time (LST)
2021       3
2022       1
2023       1
2024       1
2025       1
2026    3792
Name: count, dtype: int64


#### Missing Weather Gap Analysis

Because most missing core weather observations occur outside the available electricity-consumption period, the remaining missing observations within 2021–2025 are examined separately.

Consecutive missing timestamps are identified to determine whether the gaps are isolated or occur in longer sequences before defining an appropriate cleaning treatment.

In [94]:
# Analyze consecutive missing weather gaps within the modeling period

model_start = pd.Timestamp("2021-01-01 00:00:00")
model_end = pd.Timestamp("2025-12-31 23:00:00")

for station, df in weather_data.items():

    temp = df.copy()

    temp["weather_timestamp"] = pd.to_datetime(
        temp["Date/Time (LST)"],
        errors="coerce"
    )

    temp = temp[
        temp["weather_timestamp"].between(model_start, model_end)
    ].copy()

    missing_mask = temp[core_weather_columns].isna().any(axis=1)

    missing_times = (
        temp.loc[missing_mask, "weather_timestamp"]
        .sort_values()
        .reset_index(drop=True)
    )

    # A new gap starts whenever the difference is greater than one hour
    gap_id = missing_times.diff().ne(pd.Timedelta(hours=1)).cumsum()

    gaps = (
        missing_times.groupby(gap_id)
        .agg(["min", "max", "count"])
        .rename(columns={
            "min": "Gap_Start",
            "max": "Gap_End",
            "count": "Missing_Hours"
        })
        .reset_index(drop=True)
    )

    print("\n" + "=" * 70)
    print("Station:", station)
    print("=" * 70)
    print(gaps.to_string(index=False))


Station: TORONTO_CITY
          Gap_Start             Gap_End  Missing_Hours
2021-07-20 09:00:00 2021-07-20 13:00:00              5
2021-11-01 07:00:00 2021-11-01 07:00:00              1
2021-11-16 10:00:00 2021-11-16 10:00:00              1
2022-10-27 09:00:00 2022-10-27 13:00:00              5
2023-11-07 10:00:00 2023-11-07 14:00:00              5
2023-11-20 13:00:00 2023-11-20 13:00:00              1
2024-07-23 09:00:00 2024-07-23 13:00:00              5
2024-07-28 23:00:00 2024-07-28 23:00:00              1
2024-09-10 07:00:00 2024-09-10 07:00:00              1
2025-04-16 08:00:00 2025-04-16 13:00:00              6
2025-05-07 07:00:00 2025-05-07 07:00:00              1
2025-05-08 03:00:00 2025-05-08 03:00:00              1
2025-05-27 10:00:00 2025-05-27 10:00:00              1
2025-05-31 06:00:00 2025-05-31 06:00:00              1
2025-08-05 12:00:00 2025-08-05 12:00:00              1
2025-10-22 09:00:00 2025-10-22 13:00:00              5

Station: TORONTO_INTL_A
          Gap_Sta

In [95]:
# Compare core weather availability across stations
# during Toronto City missing periods (2021–2025)

city = weather_data["TORONTO_CITY"].copy()
intl = weather_data["TORONTO_INTL_A"].copy()

city["weather_timestamp"] = pd.to_datetime(
    city["Date/Time (LST)"],
    errors="coerce"
)

intl["weather_timestamp"] = pd.to_datetime(
    intl["Date/Time (LST)"],
    errors="coerce"
)

city_period = city[
    city["weather_timestamp"].between(model_start, model_end)
].copy()

intl_period = intl[
    intl["weather_timestamp"].between(model_start, model_end)
].copy()

city_missing_mask = city_period[core_weather_columns].isna().any(axis=1)

city_missing_times = city_period.loc[
    city_missing_mask,
    "weather_timestamp"
]

comparison = pd.DataFrame({
    "Timestamp": city_missing_times
})

comparison = comparison.merge(
    intl_period[
        ["weather_timestamp"] + core_weather_columns
    ],
    left_on="Timestamp",
    right_on="weather_timestamp",
    how="left"
)

comparison["INTL_Core_Data_Complete"] = (
    comparison[core_weather_columns]
    .notna()
    .all(axis=1)
)

print("Toronto City missing hours:", len(comparison))

print(
    "INTL A complete during those hours:",
    comparison["INTL_Core_Data_Complete"].sum()
)

print(
    "INTL A also missing:",
    (~comparison["INTL_Core_Data_Complete"]).sum()
)

comparison[
    ~comparison["INTL_Core_Data_Complete"]
]

Toronto City missing hours: 41
INTL A complete during those hours: 39
INTL A also missing: 2


,Timestamp,weather_timestamp,Temp (°C),Dew Point Temp (°C),Rel Hum (%),Stn Press (kPa),INTL_Core_Data_Complete
17,2023-11-20 13:00:00,2023-11-20 13:00:00,NaN,NaN,NaN,NaN,False
35,2025-08-05 12:00:00,2025-08-05 12:00:00,NaN,NaN,NaN,NaN,False


#### Weather Variable Classification

Weather variables are classified according to their relevance, availability, and interpretation before applying any cleaning or imputation rules.

The objective is to distinguish core meteorological measurements from metadata, quality flags, condition-dependent variables, and variables that are unavailable consistently across the selected weather stations.

This prevents structurally unavailable or condition-dependent values from being incorrectly treated as ordinary missing observations.

In [96]:
# Build weather variable availability profile

weather_variable_profile = []

for column in weather_data["TORONTO_CITY"].columns:

    row = {"Column": column}

    for station, df in weather_data.items():

        non_missing = int(df[column].notna().sum())
        missing = int(df[column].isna().sum())
        availability_pct = (non_missing / len(df)) * 100

        row[f"{station}_Available"] = non_missing
        row[f"{station}_Missing"] = missing
        row[f"{station}_Availability_%"] = round(availability_pct, 2)
        row[f"{station}_Dtype"] = str(df[column].dtype)

    weather_variable_profile.append(row)

weather_variable_profile = pd.DataFrame(weather_variable_profile)

weather_variable_profile

,Column,TORONTO_CITY_Available,TORONTO_CITY_Missing,TORONTO_CITY_Availability_%,TORONTO_CITY_Dtype,TORONTO_INTL_A_Available,TORONTO_INTL_A_Missing,TORONTO_INTL_A_Availability_%,TORONTO_INTL_A_Dtype
0,Longitude (x),52584,0,100.00,float64,52584,0,100.00,float64
1,Latitude (y),52584,0,100.00,float64,52584,0,100.00,float64
2,Station Name,52584,0,100.00,object,52584,0,100.00,object
3,Climate ID,52584,0,100.00,int64,52584,0,100.00,int64
4,Date/Time (LST),52584,0,100.00,object,52584,0,100.00,object
5,Year,52584,0,100.00,int64,52584,0,100.00,int64
6,Month,52584,0,100.00,int64,52584,0,100.00,int64
7,Day,52584,0,100.00,int64,52584,0,100.00,int64
8,Time (LST),52584,0,100.00,object,52584,0,100.00,object
9,Flag,0,52584,0.00,float64,0,52584,0.00,float64


#### Weather Variable Classification Summary

Weather variables were classified according to availability, interpretation, and relevance before applying cleaning rules.

Four continuous meteorological variables are consistently available across both stations: `Temp (°C)`, `Dew Point Temp (°C)`, `Rel Hum (%)`, and `Stn Press (kPa)`. These variables form the common core weather dataset and will be retained for subsequent analysis and modeling.

Within the common historical period (2021–2025), missing observations in these core variables are limited to short gaps of no more than six consecutive hours.

Precipitation, wind, visibility, wind chill, and textual weather conditions are not consistently available across both stations. These variables are therefore preserved in the source data but are not imputed or automatically included in the common weather feature set.

Humidex and wind chill are treated as condition-dependent variables rather than conventional missing-data problems.

Weather quality flags are retained only as source-quality metadata during cleaning and are not intended as forecasting features.

Final feature selection will be performed during the exploratory analysis and modeling stages rather than during data cleaning.

### Calendar Data Quality Assessment

The hourly calendar dataset is assessed independently before integration with electricity consumption and weather data.

The assessment focuses on completeness, duplicate records, timestamp integrity, data types, and consistency of the main temporal fields.

The calendar dataset already contains engineered temporal features, including holiday indicators, business-hour indicators, cyclical encodings, and daylight-saving-time attributes. Existing timestamp and DST processing is treated as validated upstream preprocessing and is not reconstructed in this notebook.

In [97]:
# General quality profile for calendar dataset

calendar_profile = pd.DataFrame([{
    "Rows": len(calendar),
    "Columns": len(calendar.columns),
    "Missing_Values": int(calendar.isna().sum().sum()),
    "Duplicate_Rows": int(calendar.duplicated().sum()),
    "Duplicate_Local_Timestamps": int(
        calendar["timestamp_local"].duplicated().sum()
    ),
    "Duplicate_UTC_Timestamps": int(
        calendar["timestamp_utc"].duplicated().sum()
    )
}])

calendar_profile

,Rows,Columns,Missing_Values,Duplicate_Rows,Duplicate_Local_Timestamps,Duplicate_UTC_Timestamps
0,52584,45,51168,0,0,0


In [98]:
# Identify missing values by calendar column

calendar_missing = pd.DataFrame({
    "Column": calendar.columns,
    "Missing_Count": calendar.isna().sum().values,
    "Missing_Percent": (
        calendar.isna().mean().values * 100
    ).round(2)
})

calendar_missing[
    calendar_missing["Missing_Count"] > 0
].sort_values(
    "Missing_Percent",
    ascending=False
)

,Column,Missing_Count,Missing_Percent
27,holiday_name,51168,97.31


In [99]:
# Review calendar data types

calendar_dtypes = pd.DataFrame({
    "Column": calendar.columns,
    "Dtype": calendar.dtypes.astype(str).values
})

calendar_dtypes

,Column,Dtype
0,timestamp_utc,object
1,timestamp_local,object
2,timestamp_toronto,object
3,date,object
4,year,int64
5,quarter,int64
6,month,int64
7,month_name,object
8,week_of_year,int64
9,day_of_year,int64


## Data Cleaning

Data cleaning is performed independently for electricity consumption, weather, and calendar data before regional aggregation and dataset integration.

The cleaning process follows the findings from the Data Quality Assessment and preserves the validated upstream timestamp structure.

The main cleaning rules are:

1. Original loaded datasets are not modified directly; cleaned working copies are created.
2. Consumption timestamps are converted to datetime without reconstructing the previously validated hourly structure.
3. Weather timestamps are converted to datetime while preserving the fixed-clock structure of `Date/Time (LST)`.
4. Short missing gaps in continuous weather measurements are treated only within the historical consumption period (2021–2025).
5. Variables structurally unavailable at a weather station are not imputed from the other station.
6. Condition-dependent weather variables are not treated as conventional missing values.
7. Calendar timestamp fields are converted to appropriate temporal data types.
8. Existing DST processing is preserved and not reconstructed during cleaning.
9. No regional aggregation or cross-source joins are performed during this section.
10. Cleaning results are validated before proceeding to Data Transformation and Integration.

In [100]:
# Create independent working copies for data cleaning

consumption_clean = {
    fsa: df.copy()
    for fsa, df in consumption_data.items()
}

weather_clean = {
    station: df.copy()
    for station, df in weather_data.items()
}

calendar_clean = calendar.copy()

print("Cleaning working copies created.")
print("Consumption datasets:", len(consumption_clean))
print("Weather datasets:", len(weather_clean))
print("Calendar shape:", calendar_clean.shape)

Cleaning working copies created.
Consumption datasets: 6
Weather datasets: 2
Calendar shape: (52584, 45)


### Consumption Data Cleaning

The six electricity-consumption datasets showed consistent schemas, complete hourly timestamp coverage, no missing values, and no exact duplicate records.

Differences in row counts were confirmed to result from the availability of `CUSTOMER_TYPE × PRICE_PLAN` combinations rather than missing hourly coverage.

Therefore, consumption cleaning is limited to data-type standardization and validation. Aggregation of consumption categories and construction of regional demand series are deferred to the Data Transformation and Integration stage.

In [101]:
# Standardize temporal data types in consumption datasets

for fsa, df in consumption_clean.items():

    df["DATE"] = pd.to_datetime(
        df["DATE"],
        errors="raise"
    )

    df["INTERVAL_START_TIMESTAMP"] = pd.to_datetime(
        df["INTERVAL_START_TIMESTAMP"],
        errors="raise"
    )

    df["INTERVAL_END_TIMESTAMP"] = pd.to_datetime(
        df["INTERVAL_END_TIMESTAMP"],
        errors="raise"
    )

    df["TIMESTAMP"] = pd.to_datetime(
        df["TIMESTAMP"],
        errors="raise"
    )

print("Consumption temporal columns converted successfully.")

Consumption temporal columns converted successfully.


In [102]:
# Validate consumption cleaning

consumption_clean_validation = []

for fsa, df in consumption_clean.items():

    consumption_clean_validation.append({
        "FSA": fsa,
        "Rows": len(df),
        "Missing_Values": int(df.isna().sum().sum()),
        "Duplicate_Rows": int(df.duplicated().sum()),
        "Invalid_Timestamps": int(df["TIMESTAMP"].isna().sum()),
        "Unique_Timestamps": int(df["TIMESTAMP"].nunique()),
        "Start": df["TIMESTAMP"].min(),
        "End": df["TIMESTAMP"].max()
    })

consumption_clean_validation = pd.DataFrame(
    consumption_clean_validation
)

consumption_clean_validation

,FSA,Rows,Missing_Values,Duplicate_Rows,Invalid_Timestamps,Unique_Timestamps,Start,End
0,L4T,215697,0,0,0,43824,2021-01-01,2025-12-31 23:00:00
1,M5R,101596,0,0,0,43824,2021-01-01,2025-12-31 23:00:00
2,M5S,103032,0,0,0,43824,2021-01-01,2025-12-31 23:00:00
3,M6G,203915,0,0,0,43824,2021-01-01,2025-12-31 23:00:00
4,M9R,209729,0,0,0,43824,2021-01-01,2025-12-31 23:00:00
5,M9W,212492,0,0,0,43824,2021-01-01,2025-12-31 23:00:00


### Weather Data Cleaning

Weather cleaning is performed independently for Toronto City and Toronto INTL A while preserving each station as a separate meteorological source.

Based on the Data Quality Assessment:

- Weather timestamps are standardized to datetime.
- Core continuous weather variables with short missing gaps during the 2021–2025 historical period are eligible for time-based interpolation.
- Interpolation is limited to gaps of no more than six consecutive hours.
- Structurally unavailable variables are not imputed from the other weather station.
- Condition-dependent variables such as Humidex and Wind Chill are not treated as conventional missing values.
- Weather observations outside the historical consumption period are not artificially imputed.
- Original weather datasets remain unchanged.

In [103]:
# Standardize weather timestamps

for station, df in weather_clean.items():

    df["Date/Time (LST)"] = pd.to_datetime(
        df["Date/Time (LST)"],
        errors="raise"
    )

print("Weather timestamps converted successfully.")

Weather timestamps converted successfully.


In [104]:
# Define common continuous weather variables

core_weather_columns = [
    "Temp (°C)",
    "Dew Point Temp (°C)",
    "Rel Hum (%)",
    "Stn Press (kPa)"
]

historical_start = pd.Timestamp("2021-01-01 00:00:00")
historical_end = pd.Timestamp("2025-12-31 23:00:00")

In [105]:
# Missing core weather observations before cleaning

weather_missing_before = []

for station, df in weather_clean.items():

    historical_mask = df["Date/Time (LST)"].between(
        historical_start,
        historical_end
    )

    for column in core_weather_columns:

        missing_count = int(
            df.loc[historical_mask, column].isna().sum()
        )

        weather_missing_before.append({
            "Station": station,
            "Column": column,
            "Missing_Before": missing_count
        })

weather_missing_before = pd.DataFrame(weather_missing_before)

weather_missing_before

,Station,Column,Missing_Before
0,TORONTO_CITY,Temp (°C),41
1,TORONTO_CITY,Dew Point Temp (°C),41
2,TORONTO_CITY,Rel Hum (%),41
3,TORONTO_CITY,Stn Press (kPa),41
4,TORONTO_INTL_A,Temp (°C),6
5,TORONTO_INTL_A,Dew Point Temp (°C),7
6,TORONTO_INTL_A,Rel Hum (%),7
7,TORONTO_INTL_A,Stn Press (kPa),6


#### Controlled Interpolation of Core Weather Variables

Short gaps in the four core continuous weather variables are interpolated independently within each weather station.

Interpolation is restricted to the historical consumption period (2021–2025) and only to complete missing sequences of no more than six consecutive hours.

Longer gaps, observations outside the historical consumption period, structurally unavailable weather variables, and condition-dependent variables are not imputed.

In [106]:
# Interpolate only complete missing gaps of up to 6 consecutive hours

max_gap_hours = 6

for station, df in weather_clean.items():

    historical_mask = df["Date/Time (LST)"].between(
        historical_start,
        historical_end
    )

    historical = df.loc[
        historical_mask,
        ["Date/Time (LST)"] + core_weather_columns
    ].copy()

    historical = historical.sort_values("Date/Time (LST)")
    historical = historical.set_index("Date/Time (LST)")

    for column in core_weather_columns:

        series = historical[column].copy()

        # Identify consecutive missing-value groups
        missing = series.isna()

        group_id = missing.ne(missing.shift()).cumsum()

        gap_sizes = (
            missing.groupby(group_id)
            .transform("sum")
        )

        # Candidate values using time-based interpolation
        interpolated = series.interpolate(
            method="time",
            limit_area="inside"
        )

        # Replace only missing observations belonging
        # to complete gaps of 6 hours or fewer
        eligible = (
            missing
            & (gap_sizes <= max_gap_hours)
        )

        historical.loc[eligible, column] = interpolated.loc[eligible]

    # Write cleaned historical values back to the working dataset
    historical_reset = historical.reset_index()

    for column in core_weather_columns:

        value_map = historical_reset.set_index(
            "Date/Time (LST)"
        )[column]

        df.loc[historical_mask, column] = (
            df.loc[historical_mask, "Date/Time (LST)"]
            .map(value_map)
            .values
        )

print("Controlled weather interpolation completed.")

Controlled weather interpolation completed.


In [107]:
# Validate core weather missing values after interpolation

weather_missing_after = []

for station, df in weather_clean.items():

    historical_mask = df["Date/Time (LST)"].between(
        historical_start,
        historical_end
    )

    for column in core_weather_columns:

        missing_after = int(
            df.loc[historical_mask, column].isna().sum()
        )

        weather_missing_after.append({
            "Station": station,
            "Column": column,
            "Missing_After": missing_after
        })

weather_missing_after = pd.DataFrame(weather_missing_after)

weather_interpolation_validation = weather_missing_before.merge(
    weather_missing_after,
    on=["Station", "Column"]
)

weather_interpolation_validation["Values_Imputed"] = (
    weather_interpolation_validation["Missing_Before"]
    - weather_interpolation_validation["Missing_After"]
)

weather_interpolation_validation

,Station,Column,Missing_Before,Missing_After,Values_Imputed
0,TORONTO_CITY,Temp (°C),41,0,41
1,TORONTO_CITY,Dew Point Temp (°C),41,0,41
2,TORONTO_CITY,Rel Hum (%),41,0,41
3,TORONTO_CITY,Stn Press (kPa),41,0,41
4,TORONTO_INTL_A,Temp (°C),6,0,6
5,TORONTO_INTL_A,Dew Point Temp (°C),7,0,7
6,TORONTO_INTL_A,Rel Hum (%),7,0,7
7,TORONTO_INTL_A,Stn Press (kPa),6,0,6


#### Core Weather Interpolation Validation

Controlled time-based interpolation successfully resolved all short missing gaps in the four core continuous weather variables during the 2021–2025 historical period.

For Toronto City, 41 missing observations were imputed independently for temperature, dew point temperature, relative humidity, and station pressure.

For Toronto INTL A, 6 missing temperature observations, 7 dew point observations, 7 relative humidity observations, and 6 station-pressure observations were imputed.

All eligible gaps were six consecutive hours or shorter. No timestamps were removed, no values from one weather station were used to populate the other station, and no long missing periods outside the historical consumption window were imputed.

Variables that are structurally unavailable or condition-dependent remain unchanged.

In [108]:
# Final structural validation of cleaned weather datasets

weather_clean_validation = []

for station, df in weather_clean.items():

    historical_mask = df["Date/Time (LST)"].between(
        historical_start,
        historical_end
    )

    historical_df = df.loc[historical_mask]

    weather_clean_validation.append({
        "Station": station,
        "Historical_Rows": len(historical_df),
        "Unique_Timestamps": historical_df["Date/Time (LST)"].nunique(),
        "Duplicate_Timestamps": int(
            historical_df["Date/Time (LST)"].duplicated().sum()
        ),
        "Core_Missing_After": int(
            historical_df[core_weather_columns]
            .isna()
            .sum()
            .sum()
        ),
        "Start": historical_df["Date/Time (LST)"].min(),
        "End": historical_df["Date/Time (LST)"].max()
    })

weather_clean_validation = pd.DataFrame(
    weather_clean_validation
)

weather_clean_validation

,Station,Historical_Rows,Unique_Timestamps,Duplicate_Timestamps,Core_Missing_After,Start,End
0,TORONTO_CITY,43824,43824,0,0,2021-01-01,2025-12-31 23:00:00
1,TORONTO_INTL_A,43824,43824,0,0,2021-01-01,2025-12-31 23:00:00


### Calendar Data Cleaning

Calendar cleaning focuses on temporal data-type standardization while preserving the existing calendar features and previously validated DST structure.

The `timestamp_utc`, `timestamp_local`, `timestamp_toronto`, and `date` fields were originally loaded as object data types and are converted to appropriate temporal types.

Binary calendar indicators remain encoded as 0/1 integers.

Missing values in `holiday_name` are preserved because they represent timestamps that are not associated with a named public holiday.

No timezone-shift transformation or cross-source integration is performed at this stage.

In [109]:
# Standardize calendar temporal data types

calendar_clean["timestamp_utc"] = pd.to_datetime(
    calendar_clean["timestamp_utc"],
    errors="raise"
)

calendar_clean["timestamp_local"] = pd.to_datetime(
    calendar_clean["timestamp_local"],
    errors="raise"
)

calendar_clean["timestamp_toronto"] = pd.to_datetime(
    calendar_clean["timestamp_toronto"],
    errors="raise"
)

calendar_clean["date"] = pd.to_datetime(
    calendar_clean["date"],
    errors="raise"
)

print("Calendar temporal columns converted successfully.")

Calendar temporal columns converted successfully.


C:\Users\ASUS\AppData\Local\Temp\ipykernel_40584\2339055946.py:13: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  calendar_clean["timestamp_toronto"] = pd.to_datetime(


In [110]:
# Validate calendar cleaning

calendar_clean_validation = pd.DataFrame([{
    "Rows": len(calendar_clean),
    "Columns": len(calendar_clean.columns),
    "Duplicate_Rows": int(calendar_clean.duplicated().sum()),
    "Duplicate_UTC_Timestamps": int(
        calendar_clean["timestamp_utc"].duplicated().sum()
    ),
    "Duplicate_Local_Timestamps": int(
        calendar_clean["timestamp_local"].duplicated().sum()
    ),
    "Invalid_UTC_Timestamps": int(
        calendar_clean["timestamp_utc"].isna().sum()
    ),
    "Invalid_Local_Timestamps": int(
        calendar_clean["timestamp_local"].isna().sum()
    ),
    "Start_UTC": calendar_clean["timestamp_utc"].min(),
    "End_UTC": calendar_clean["timestamp_utc"].max()
}])

calendar_clean_validation

,Rows,Columns,Duplicate_Rows,Duplicate_UTC_Timestamps,Duplicate_Local_Timestamps,Invalid_UTC_Timestamps,Invalid_Local_Timestamps,Start_UTC,End_UTC
0,52584,45,0,0,0,0,0,2021-01-01 05:00:00+00:00,2027-01-01 04:00:00+00:00


### Data Cleaning Summary

Data cleaning was completed independently for electricity consumption, weather, and calendar data without modifying the original loaded datasets.

Electricity-consumption data required temporal data-type standardization only. All six FSA datasets retained complete hourly timestamp coverage from 2021 through 2025, with no missing values, duplicate rows, or invalid timestamps.

Weather data from both stations retained complete hourly coverage during 2021–2025. Short gaps of no more than six consecutive hours in the four common continuous meteorological variables were resolved using controlled time-based interpolation. No observations were removed, no weather stations were mixed, and structurally unavailable or condition-dependent variables were not artificially imputed.

Calendar temporal fields were converted to appropriate datetime types while preserving the existing calendar and DST structure. Missing `holiday_name` values were retained as semantically expected values.

The cleaned datasets are now structurally ready for transformation and integration. Regional aggregation, weather-station assignment, calendar alignment, and cross-source joins are intentionally deferred to the next stage.

## Data Transformation and Integration

The cleaned datasets are transformed into regional hourly datasets suitable for subsequent exploratory analysis and forecasting.

The transformation pipeline follows a controlled sequence:

1. Aggregate electricity-consumption categories to one hourly observation per FSA.
2. Validate the resulting FSA-level hourly series.
3. Aggregate the selected FSAs into the two defined study regions.
4. Assign the corresponding weather station to each region.
5. Integrate regional electricity demand with weather data.
6. Align and integrate calendar features using the project's defined timestamp strategy.
7. Validate temporal coverage, uniqueness, missing values, and row counts after each transformation.

No exploratory analysis or forecasting is performed during this stage.

### Hourly Consumption Aggregation by FSA

Electricity consumption is originally segmented by `CUSTOMER_TYPE` and `PRICE_PLAN`, resulting in multiple records for the same FSA and timestamp.

These components are aggregated by summing `TOTAL_CONSUMPTION` across all available customer-type and price-plan combinations for each `FSA + TIMESTAMP`.

Missing category combinations are not artificially created or assigned zero values. Only observations available in the source data are included in the hourly total.

In [111]:
# Aggregate consumption categories to one hourly observation per FSA

consumption_hourly = {}

for fsa, df in consumption_clean.items():

    hourly = (
        df.groupby(
            ["FSA", "TIMESTAMP"],
            as_index=False
        )
        .agg(
            TOTAL_CONSUMPTION=("TOTAL_CONSUMPTION", "sum"),
            PREMISE_COUNT=("PREMISE_COUNT", "sum")
        )
        .sort_values("TIMESTAMP")
        .reset_index(drop=True)
    )

    consumption_hourly[fsa] = hourly

print("Hourly consumption datasets created:", len(consumption_hourly))

Hourly consumption datasets created: 6


In [112]:
# Validate hourly FSA aggregation

fsa_hourly_validation = []

for fsa, df in consumption_hourly.items():

    fsa_hourly_validation.append({
        "FSA": fsa,
        "Rows": len(df),
        "Unique_Timestamps": df["TIMESTAMP"].nunique(),
        "Duplicate_Timestamps": int(
            df["TIMESTAMP"].duplicated().sum()
        ),
        "Missing_Consumption": int(
            df["TOTAL_CONSUMPTION"].isna().sum()
        ),
        "Min_Consumption": df["TOTAL_CONSUMPTION"].min(),
        "Max_Consumption": df["TOTAL_CONSUMPTION"].max(),
        "Start": df["TIMESTAMP"].min(),
        "End": df["TIMESTAMP"].max()
    })

fsa_hourly_validation = pd.DataFrame(
    fsa_hourly_validation
)

fsa_hourly_validation

,FSA,Rows,Unique_Timestamps,Duplicate_Timestamps,Missing_Consumption,Min_Consumption,Max_Consumption,Start,End
0,L4T,43824,43824,0,0,5864.4,27796.6,2021-01-01,2025-12-31 23:00:00
1,M5R,43824,43824,0,0,3721.5,16999.0,2021-01-01,2025-12-31 23:00:00
2,M5S,43824,43824,0,0,2060.4,7622.4,2021-01-01,2025-12-31 23:00:00
3,M6G,43824,43824,0,0,4954.9,27460.1,2021-01-01,2025-12-31 23:00:00
4,M9R,43824,43824,0,0,1377.7,16243.6,2021-01-01,2025-12-31 23:00:00
5,M9W,43824,43824,0,0,4361.1,33712.2,2021-01-01,2025-12-31 23:00:00


In [113]:
# Validate conservation of total consumption during aggregation

aggregation_conservation = []

for fsa in consumption_clean.keys():

    original_total = consumption_clean[fsa][
        "TOTAL_CONSUMPTION"
    ].sum()

    aggregated_total = consumption_hourly[fsa][
        "TOTAL_CONSUMPTION"
    ].sum()

    difference = aggregated_total - original_total

    aggregation_conservation.append({
        "FSA": fsa,
        "Original_Total": original_total,
        "Aggregated_Total": aggregated_total,
        "Difference": difference,
        "Conservation_OK": abs(difference) < 1e-6
    })

aggregation_conservation = pd.DataFrame(
    aggregation_conservation
)

aggregation_conservation

,FSA,Original_Total,Aggregated_Total,Difference,Conservation_OK
0,L4T,509146142.9,509146142.9,0.000000e+00,True
1,M5R,335781093.0,335781093.0,0.000000e+00,True
2,M5S,180061857.9,180061857.9,2.980232e-08,True
3,M6G,503455261.6,503455261.6,-5.960464e-08,True
4,M9R,242873032.0,242873032.0,0.000000e+00,True
5,M9W,540674797.2,540674797.2,0.000000e+00,True


### Regional Consumption Aggregation

The six FSA-level hourly consumption series are aggregated into the two study regions defined in the project methodology.

- **Downtown:** M5S + M5R + M6G
- **Airport-West:** L4T + M9W + M9R

Because each FSA contains exactly one observation for each of the 43,824 hourly timestamps from 2021 through 2025, regional electricity demand can be calculated by summing the three corresponding FSA-level hourly consumption values.

This transformation produces one hourly electricity-demand target for each region.

In [114]:
# Define regional FSA groups

region_fsa_map = {
    "DOWNTOWN": ["M5S", "M5R", "M6G"],
    "AIRPORT_WEST": ["L4T", "M9W", "M9R"]
}

region_fsa_map

{'DOWNTOWN': ['M5S', 'M5R', 'M6G'], 'AIRPORT_WEST': ['L4T', 'M9W', 'M9R']}

In [115]:
# Aggregate FSA-level hourly consumption into regional demand

regional_consumption = {}

for region, fsas in region_fsa_map.items():

    region_data = pd.concat(
        [
            consumption_hourly[fsa][
                ["FSA", "TIMESTAMP", "TOTAL_CONSUMPTION"]
            ]
            for fsa in fsas
        ],
        ignore_index=True
    )

    region_hourly = (
        region_data
        .groupby(
            "TIMESTAMP",
            as_index=False
        )
        .agg(
            TOTAL_CONSUMPTION=("TOTAL_CONSUMPTION", "sum"),
            FSA_COUNT=("FSA", "nunique")
        )
        .sort_values("TIMESTAMP")
        .reset_index(drop=True)
    )

    region_hourly.insert(0, "REGION", region)

    regional_consumption[region] = region_hourly

print("Regional consumption datasets created:")
for region, df in regional_consumption.items():
    print(region, df.shape)

Regional consumption datasets created:
DOWNTOWN (43824, 4)
AIRPORT_WEST (43824, 4)


In [116]:
# Validate regional hourly consumption

regional_validation = []

for region, df in regional_consumption.items():

    regional_validation.append({
        "Region": region,
        "Rows": len(df),
        "Unique_Timestamps": df["TIMESTAMP"].nunique(),
        "Duplicate_Timestamps": int(
            df["TIMESTAMP"].duplicated().sum()
        ),
        "Missing_Consumption": int(
            df["TOTAL_CONSUMPTION"].isna().sum()
        ),
        "Min_FSA_Count": int(df["FSA_COUNT"].min()),
        "Max_FSA_Count": int(df["FSA_COUNT"].max()),
        "Start": df["TIMESTAMP"].min(),
        "End": df["TIMESTAMP"].max()
    })

regional_validation = pd.DataFrame(regional_validation)

regional_validation

,Region,Rows,Unique_Timestamps,Duplicate_Timestamps,Missing_Consumption,Min_FSA_Count,Max_FSA_Count,Start,End
0,DOWNTOWN,43824,43824,0,0,3,3,2021-01-01,2025-12-31 23:00:00
1,AIRPORT_WEST,43824,43824,0,0,3,3,2021-01-01,2025-12-31 23:00:00


In [117]:
# Validate conservation of consumption during regional aggregation

regional_conservation = []

for region, fsas in region_fsa_map.items():

    expected_total = sum(
        consumption_hourly[fsa]["TOTAL_CONSUMPTION"].sum()
        for fsa in fsas
    )

    regional_total = (
        regional_consumption[region]["TOTAL_CONSUMPTION"].sum()
    )

    difference = regional_total - expected_total

    regional_conservation.append({
        "Region": region,
        "Expected_Total": expected_total,
        "Regional_Total": regional_total,
        "Difference": difference,
        "Conservation_OK": abs(difference) < 1e-6
    })

regional_conservation = pd.DataFrame(regional_conservation)

regional_conservation

,Region,Expected_Total,Regional_Total,Difference,Conservation_OK
0,DOWNTOWN,1.019298e+09,1.019298e+09,0.0,True
1,AIRPORT_WEST,1.292694e+09,1.292694e+09,0.0,True


### Weather Station Assignment and Integration

Each regional electricity-demand series is associated with its designated weather station according to the geographic mapping defined in the project methodology:

- **Downtown → Toronto City**
- **Airport-West → Toronto INTL A**

Weather stations are not averaged or combined. Each region retains the meteorological observations from its assigned station.

Electricity-consumption `TIMESTAMP` and weather `Date/Time (LST)` follow the same validated fixed-clock hourly structure and are therefore joined directly at this stage.

Weather variables that are structurally unavailable at the assigned station remain unavailable and are not populated using observations from the other station.

In [118]:
# Define regional weather-station mapping

region_weather_map = {
    "DOWNTOWN": "TORONTO_CITY",
    "AIRPORT_WEST": "TORONTO_INTL_A"
}

region_weather_map

{'DOWNTOWN': 'TORONTO_CITY', 'AIRPORT_WEST': 'TORONTO_INTL_A'}

In [119]:
# Prepare regional weather datasets for the common historical period

regional_weather = {}

for region, station in region_weather_map.items():

    weather_region = weather_clean[station].copy()

    weather_region = weather_region[
        weather_region["Date/Time (LST)"].between(
            historical_start,
            historical_end
        )
    ].copy()

    weather_region = (
        weather_region
        .sort_values("Date/Time (LST)")
        .reset_index(drop=True)
    )

    regional_weather[region] = weather_region

for region, df in regional_weather.items():
    print(
        region,
        "| Station:", region_weather_map[region],
        "| Shape:", df.shape
    )

DOWNTOWN | Station: TORONTO_CITY | Shape: (43824, 33)
AIRPORT_WEST | Station: TORONTO_INTL_A | Shape: (43824, 33)


In [120]:
# Validate timestamp alignment before weather integration

weather_alignment_validation = []

for region in regional_consumption.keys():

    consumption_ts = (
        regional_consumption[region]["TIMESTAMP"]
        .sort_values()
        .reset_index(drop=True)
    )

    weather_ts = (
        regional_weather[region]["Date/Time (LST)"]
        .sort_values()
        .reset_index(drop=True)
    )

    exact_match = consumption_ts.equals(weather_ts)

    consumption_only = len(
        set(consumption_ts) - set(weather_ts)
    )

    weather_only = len(
        set(weather_ts) - set(consumption_ts)
    )

    weather_alignment_validation.append({
        "Region": region,
        "Consumption_Timestamps": len(consumption_ts),
        "Weather_Timestamps": len(weather_ts),
        "Exact_Sequence_Match": exact_match,
        "Consumption_Only": consumption_only,
        "Weather_Only": weather_only
    })

weather_alignment_validation = pd.DataFrame(
    weather_alignment_validation
)

weather_alignment_validation

,Region,Consumption_Timestamps,Weather_Timestamps,Exact_Sequence_Match,Consumption_Only,Weather_Only
0,DOWNTOWN,43824,43824,True,0,0
1,AIRPORT_WEST,43824,43824,True,0,0


#### Minimal Weather Feature Integration

Only meteorological variables required by the defined project scope are retained during regional integration.

To minimize unnecessary storage, processing, missing-data handling, and downstream model complexity, weather variables without a defined analytical purpose are excluded from the integrated datasets.

The regional weather feature sets are:

- **Downtown / Toronto City:** temperature, relative humidity, and precipitation.
- **Airport-West / Toronto INTL A:** temperature, relative humidity, wind speed, and visibility.

Variables such as dew point temperature, station pressure, Humidex, wind chill, textual weather conditions, wind direction, source metadata, quality flags, and redundant temporal fields are not propagated into the regional analytical datasets.

Variables unavailable at one station are not reconstructed, cross-station imputed, or added merely to force identical schemas.

This produces intentionally different weather schemas for the two regional datasets, reflecting the meteorological information actually available and required for each regional analysis.

In [121]:
# Define minimal common weather feature set

weather_features = {
    "TORONTO_CITY": [
        "Temp (°C)",
        "Rel Hum (%)"
    ],

    "TORONTO_INTL_A": [
        "Temp (°C)",
        "Rel Hum (%)"
    ]
}

weather_features

{'TORONTO_CITY': ['Temp (°C)', 'Rel Hum (%)'],
 'TORONTO_INTL_A': ['Temp (°C)', 'Rel Hum (%)']}

In [122]:
# Integrate regional consumption with selected weather features

regional_consumption_weather = {}

for region, station in region_weather_map.items():

    consumption_df = regional_consumption[region].copy()

    selected_weather_columns = (
        ["Date/Time (LST)"]
        + weather_features[station]
    )

    weather_df = regional_weather[region][
        selected_weather_columns
    ].copy()

    integrated = consumption_df.merge(
        weather_df,
        left_on="TIMESTAMP",
        right_on="Date/Time (LST)",
        how="left",
        validate="one_to_one"
    )

    integrated = integrated.drop(
        columns=["Date/Time (LST)"]
    )

    regional_consumption_weather[region] = integrated

for region, df in regional_consumption_weather.items():
    print(region)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print()

DOWNTOWN
Shape: (43824, 6)
Columns: ['REGION', 'TIMESTAMP', 'TOTAL_CONSUMPTION', 'FSA_COUNT', 'Temp (°C)', 'Rel Hum (%)']

AIRPORT_WEST
Shape: (43824, 6)
Columns: ['REGION', 'TIMESTAMP', 'TOTAL_CONSUMPTION', 'FSA_COUNT', 'Temp (°C)', 'Rel Hum (%)']



#### Weather Integration Summary

Regional electricity-demand data were integrated with the assigned weather station using a one-to-one hourly timestamp match.

Only temperature and relative humidity were propagated into the base analytical datasets. These variables are available for both regions, were successfully cleaned for the 2021–2025 historical period, and have a direct analytical rationale for electricity-demand forecasting.

Other available meteorological variables, including precipitation, wind, visibility, dew point, station pressure, Humidex, wind chill, and textual weather conditions, remain preserved in the cleaned source datasets but are not included in the base integrated datasets at this stage.

This minimizes unnecessary data volume and downstream model complexity while preserving the option to evaluate additional weather variables later if justified during exploratory analysis.

In [123]:
# Validate regional consumption-weather integration

weather_merge_validation = []

for region, df in regional_consumption_weather.items():

    weather_merge_validation.append({
        "Region": region,
        "Rows": len(df),
        "Unique_Timestamps": df["TIMESTAMP"].nunique(),
        "Duplicate_Timestamps": int(
            df["TIMESTAMP"].duplicated().sum()
        ),
        "Missing_Consumption": int(
            df["TOTAL_CONSUMPTION"].isna().sum()
        ),
        "Missing_Temperature": int(
            df["Temp (°C)"].isna().sum()
        ),
        "Missing_Humidity": int(
            df["Rel Hum (%)"].isna().sum()
        ),
        "Min_FSA_Count": int(df["FSA_COUNT"].min()),
        "Max_FSA_Count": int(df["FSA_COUNT"].max()),
        "Start": df["TIMESTAMP"].min(),
        "End": df["TIMESTAMP"].max()
    })

weather_merge_validation = pd.DataFrame(
    weather_merge_validation
)

weather_merge_validation

,Region,Rows,Unique_Timestamps,Duplicate_Timestamps,Missing_Consumption,Missing_Temperature,Missing_Humidity,Min_FSA_Count,Max_FSA_Count,Start,End
0,DOWNTOWN,43824,43824,0,0,0,0,3,3,2021-01-01,2025-12-31 23:00:00
1,AIRPORT_WEST,43824,43824,0,0,0,0,3,3,2021-01-01,2025-12-31 23:00:00


### Calendar Temporal Alignment

The calendar dataset contains timezone-aware temporal information, including UTC and Toronto-local timestamps, while the electricity-consumption and weather datasets use the project's previously validated fixed-clock hourly structure.

Before integrating calendar features, the temporal relationship between the regional fixed-clock timestamps and the calendar UTC timeline is explicitly validated.

According to the project methodology, the candidate integration key is constructed as:

`fixed-clock timestamp + 5 hours → calendar timestamp_utc`

The transformation is validated before any calendar merge is performed to prevent silent temporal displacement or daylight-saving-time misalignment.

In [124]:
# Create candidate calendar join keys without modifying source timestamps

calendar_alignment_check = {}

for region, df in regional_consumption_weather.items():

    check = df[["TIMESTAMP"]].copy()

    check["candidate_utc"] = (
        check["TIMESTAMP"]
        + pd.Timedelta(hours=5)
    )

    check["candidate_utc"] = (
        check["candidate_utc"]
        .dt.tz_localize("UTC")
    )

    calendar_alignment_check[region] = check

print("Candidate UTC join keys created.")

Candidate UTC join keys created.


In [125]:
# Validate candidate UTC keys against calendar timestamps

calendar_utc_historical = calendar_clean[
    calendar_clean["timestamp_utc"].between(
        pd.Timestamp("2021-01-01 05:00:00", tz="UTC"),
        pd.Timestamp("2026-01-01 04:00:00", tz="UTC")
    )
]["timestamp_utc"]

calendar_alignment_validation = []

for region, check in calendar_alignment_check.items():

    candidate = check["candidate_utc"]

    matched = candidate.isin(calendar_utc_historical)

    calendar_alignment_validation.append({
        "Region": region,
        "Regional_Timestamps": len(candidate),
        "Matched_Calendar_Timestamps": int(matched.sum()),
        "Unmatched_Timestamps": int((~matched).sum()),
        "Match_Percent": round(
            matched.mean() * 100,
            4
        ),
        "First_Candidate_UTC": candidate.min(),
        "Last_Candidate_UTC": candidate.max()
    })

calendar_alignment_validation = pd.DataFrame(
    calendar_alignment_validation
)

calendar_alignment_validation

,Region,Regional_Timestamps,Matched_Calendar_Timestamps,Unmatched_Timestamps,Match_Percent,First_Candidate_UTC,Last_Candidate_UTC
0,DOWNTOWN,43824,43824,0,100.0,2021-01-01 05:00:00+00:00,2026-01-01 04:00:00+00:00
1,AIRPORT_WEST,43824,43824,0,100.0,2021-01-01 05:00:00+00:00,2026-01-01 04:00:00+00:00


#### Calendar Alignment Semantic Validation

Although the candidate UTC join key achieved a 100% structural match with the calendar timeline, timestamp existence alone does not guarantee correct calendar-hour attribution.

Representative winter, summer, spring-forward, and fall-back timestamps are therefore inspected before integration to confirm that the fixed-clock regional timestamps map to the intended calendar characteristics.

In [126]:
# Inspect representative timestamps before calendar integration

test_timestamps = pd.to_datetime([
    "2025-01-15 12:00:00",  # Normal winter day
    "2025-07-15 12:00:00",  # Normal summer day
    "2025-03-09 02:00:00",  # Spring-forward day
    "2025-11-02 01:00:00"   # Fall-back day
])

calendar_semantic_check = pd.DataFrame({
    "Original_TIMESTAM​P": test_timestamps
})

calendar_semantic_check["Candidate_UTC"] = (
    calendar_semantic_check["Original_TIMESTAM​P"]
    + pd.Timedelta(hours=5)
).dt.tz_localize("UTC")

calendar_semantic_check = calendar_semantic_check.merge(
    calendar_clean[
        [
            "timestamp_utc",
            "timestamp_local",
            "timestamp_toronto",
            "date",
            "hour",
            "weekday_name",
            "is_daylight_saving_time",
            "is_dst_transition_day",
            "is_spring_forward_day",
            "is_fall_back_day",
            "utc_offset_hours"
        ]
    ],
    left_on="Candidate_UTC",
    right_on="timestamp_utc",
    how="left",
    validate="one_to_one"
)

calendar_semantic_check

,Original_TIMESTAM​P,Candidate_UTC,timestamp_utc,timestamp_local,timestamp_toronto,date,hour,weekday_name,is_daylight_saving_time,is_dst_transition_day,is_spring_forward_day,is_fall_back_day,utc_offset_hours
0,2025-01-15 12:00:00,2025-01-15 17:00:00+00:00,2025-01-15 17:00:00+00:00,2025-01-15 12:00:00,2025-01-15 12:00:00-05:00,2025-01-15,12,Wednesday,0,0,0,0,-5
1,2025-07-15 12:00:00,2025-07-15 17:00:00+00:00,2025-07-15 17:00:00+00:00,2025-07-15 12:00:00,2025-07-15 13:00:00-04:00,2025-07-15,12,Tuesday,1,0,0,0,-4
2,2025-03-09 02:00:00,2025-03-09 07:00:00+00:00,2025-03-09 07:00:00+00:00,2025-03-09 02:00:00,2025-03-09 03:00:00-04:00,2025-03-09,2,Sunday,1,1,1,0,-4
3,2025-11-02 01:00:00,2025-11-02 06:00:00+00:00,2025-11-02 06:00:00+00:00,2025-11-02 01:00:00,2025-11-02 01:00:00-05:00,2025-11-02,1,Sunday,0,1,0,1,-5


In [127]:
calendar_features = [
    "timestamp_utc",
    "date",
    "year",
    "month",
    "hour",
    "weekday",
    "is_weekend",
    "is_workday",
    "is_public_holiday",
    "holiday_name",
    "is_daylight_saving_time",
    "is_dst_transition_day"
]

#### Calendar Integration

Calendar integration uses the project's validated fixed-clock temporal convention.

Regional consumption and weather timestamps follow a fixed UTC−5 analytical clock. A UTC join key is therefore constructed by adding five hours to the regional `TIMESTAMP` and matching it against `calendar.timestamp_utc`.

Semantic validation across winter, summer, spring-forward, and fall-back timestamps confirmed that this transformation preserves the intended fixed-clock `hour` while the calendar dataset independently retains Toronto daylight-saving-time information.

Only a minimal set of calendar variables relevant to the forecasting and peak-risk objectives is propagated into the base analytical datasets. Additional derived temporal features may be evaluated later during feature engineering rather than duplicated at integration time.

In [128]:
# Define minimal calendar feature set

calendar_features = [
    "timestamp_utc",
    "date",
    "year",
    "month",
    "hour",
    "weekday",
    "is_weekend",
    "is_workday",
    "is_public_holiday",
    "holiday_name",
    "is_daylight_saving_time",
    "is_dst_transition_day"
]

In [129]:
# Integrate calendar features into regional datasets

regional_master = {}

calendar_subset = calendar_clean[
    calendar_features
].copy()

for region, df in regional_consumption_weather.items():

    master = df.copy()

    # Build UTC join key from the validated fixed-clock timestamp
    master["calendar_join_utc"] = (
        master["TIMESTAMP"]
        + pd.Timedelta(hours=5)
    ).dt.tz_localize("UTC")

    master = master.merge(
        calendar_subset,
        left_on="calendar_join_utc",
        right_on="timestamp_utc",
        how="left",
        validate="one_to_one"
    )

    regional_master[region] = master

for region, df in regional_master.items():
    print(region, "| Shape:", df.shape)

DOWNTOWN | Shape: (43824, 19)
AIRPORT_WEST | Shape: (43824, 19)


In [130]:
# Validate calendar integration

calendar_merge_validation = []

for region, df in regional_master.items():

    calendar_merge_validation.append({
        "Region": region,
        "Rows": len(df),
        "Unique_Timestamps": df["TIMESTAMP"].nunique(),
        "Duplicate_Timestamps": int(
            df["TIMESTAMP"].duplicated().sum()
        ),
        "Missing_Consumption": int(
            df["TOTAL_CONSUMPTION"].isna().sum()
        ),
        "Missing_Temperature": int(
            df["Temp (°C)"].isna().sum()
        ),
        "Missing_Humidity": int(
            df["Rel Hum (%)"].isna().sum()
        ),
        "Missing_Calendar_Join": int(
            df["timestamp_utc"].isna().sum()
        ),
        "Start": df["TIMESTAMP"].min(),
        "End": df["TIMESTAMP"].max()
    })

calendar_merge_validation = pd.DataFrame(
    calendar_merge_validation
)

calendar_merge_validation

,Region,Rows,Unique_Timestamps,Duplicate_Timestamps,Missing_Consumption,Missing_Temperature,Missing_Humidity,Missing_Calendar_Join,Start,End
0,DOWNTOWN,43824,43824,0,0,0,0,0,2021-01-01,2025-12-31 23:00:00
1,AIRPORT_WEST,43824,43824,0,0,0,0,0,2021-01-01,2025-12-31 23:00:00


In [131]:
# Validate calendar feature consistency with fixed-clock timestamp

calendar_consistency_validation = []

for region, df in regional_master.items():

    hour_mismatch = (
        df["TIMESTAMP"].dt.hour != df["hour"]
    ).sum()

    date_mismatch = (
        df["TIMESTAMP"].dt.normalize()
        != df["date"]
    ).sum()

    year_mismatch = (
        df["TIMESTAMP"].dt.year != df["year"]
    ).sum()

    month_mismatch = (
        df["TIMESTAMP"].dt.month != df["month"]
    ).sum()

    weekday_mismatch = (
        df["TIMESTAMP"].dt.weekday != df["weekday"]
    ).sum()

    calendar_consistency_validation.append({
        "Region": region,
        "Hour_Mismatch": int(hour_mismatch),
        "Date_Mismatch": int(date_mismatch),
        "Year_Mismatch": int(year_mismatch),
        "Month_Mismatch": int(month_mismatch),
        "Weekday_Mismatch": int(weekday_mismatch)
    })

calendar_consistency_validation = pd.DataFrame(
    calendar_consistency_validation
)

calendar_consistency_validation

,Region,Hour_Mismatch,Date_Mismatch,Year_Mismatch,Month_Mismatch,Weekday_Mismatch
0,DOWNTOWN,0,0,0,0,0
1,AIRPORT_WEST,0,0,0,0,0


### 9.5 Final Master Dataset Construction

Following successful consumption, weather, and calendar integration, the regional datasets are reduced to the variables required for subsequent exploratory analysis and modeling.

Temporary integration keys and quality-control variables used during data transformation are removed from the final analytical datasets.

The final datasets preserve:

- the hourly analytical timestamp;
- regional electricity consumption;
- temperature and relative humidity;
- essential calendar variables;
- holiday and daylight-saving-time indicators.

Additional derived predictors, including lagged demand, rolling statistics, cyclical encodings, and the peak-risk target, are intentionally not created at this stage. These transformations belong to subsequent feature-engineering and modeling stages and must respect the temporal train/test structure to prevent data leakage.

In [132]:
# Remove technical integration and validation columns

columns_to_remove = [
    "FSA_COUNT",
    "calendar_join_utc",
    "timestamp_utc"
]

master_datasets = {}

for region, df in regional_master.items():

    master = df.drop(
        columns=columns_to_remove
    ).copy()

    master = (
        master
        .sort_values("TIMESTAMP")
        .reset_index(drop=True)
    )

    master_datasets[region] = master

for region, df in master_datasets.items():
    print(region)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print()

DOWNTOWN
Shape: (43824, 16)
Columns: ['REGION', 'TIMESTAMP', 'TOTAL_CONSUMPTION', 'Temp (°C)', 'Rel Hum (%)', 'date', 'year', 'month', 'hour', 'weekday', 'is_weekend', 'is_workday', 'is_public_holiday', 'holiday_name', 'is_daylight_saving_time', 'is_dst_transition_day']

AIRPORT_WEST
Shape: (43824, 16)
Columns: ['REGION', 'TIMESTAMP', 'TOTAL_CONSUMPTION', 'Temp (°C)', 'Rel Hum (%)', 'date', 'year', 'month', 'hour', 'weekday', 'is_weekend', 'is_workday', 'is_public_holiday', 'holiday_name', 'is_daylight_saving_time', 'is_dst_transition_day']



In [133]:
# Final validation of analytical master datasets

master_validation = []

for region, df in master_datasets.items():

    master_validation.append({
        "Region": region,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Unique_Timestamps": df["TIMESTAMP"].nunique(),
        "Duplicate_Timestamps": int(
            df["TIMESTAMP"].duplicated().sum()
        ),
        "Missing_Consumption": int(
            df["TOTAL_CONSUMPTION"].isna().sum()
        ),
        "Missing_Temperature": int(
            df["Temp (°C)"].isna().sum()
        ),
        "Missing_Humidity": int(
            df["Rel Hum (%)"].isna().sum()
        ),
        "Missing_Calendar_Core": int(
            df[
                [
                    "date",
                    "year",
                    "month",
                    "hour",
                    "weekday",
                    "is_weekend",
                    "is_workday",
                    "is_public_holiday",
                    "is_daylight_saving_time",
                    "is_dst_transition_day"
                ]
            ].isna().sum().sum()
        ),
        "Start": df["TIMESTAMP"].min(),
        "End": df["TIMESTAMP"].max()
    })

master_validation = pd.DataFrame(master_validation)

master_validation

,Region,Rows,Columns,Unique_Timestamps,Duplicate_Timestamps,Missing_Consumption,Missing_Temperature,Missing_Humidity,Missing_Calendar_Core,Start,End
0,DOWNTOWN,43824,16,43824,0,0,0,0,0,2021-01-01,2025-12-31 23:00:00
1,AIRPORT_WEST,43824,16,43824,0,0,0,0,0,2021-01-01,2025-12-31 23:00:00


### Export Final Analytical Datasets

Following successful structural, temporal, and missing-data validation, the two regional analytical datasets are exported to the processed-data directory.

The exported datasets represent the final output of the data cleaning, transformation, and integration pipeline.

They contain one observation per region and hour for the complete 2021–2025 study period and will serve as the input datasets for the subsequent exploratory data analysis and modeling stages.

No exploratory analysis, predictive feature engineering, peak-risk target construction, or model-specific transformations are performed in this notebook.

In [134]:
# Export final analytical datasets

output_files = {
    "DOWNTOWN": DATA_PATH / "master_hourly_downtown.csv",
    "AIRPORT_WEST": DATA_PATH / "master_hourly_airport_west.csv"
}

for region, output_path in output_files.items():

    master_datasets[region].to_csv(
        output_path,
        index=False
    )

    print(
        region,
        "saved to:",
        output_path
    )

DOWNTOWN saved to: ..\data\processed\master_hourly_downtown.csv
AIRPORT_WEST saved to: ..\data\processed\master_hourly_airport_west.csv


In [135]:
# Verify exported analytical datasets

for region, output_path in output_files.items():

    verification_df = pd.read_csv(
        output_path,
        nrows=5
    )

    print("=" * 60)
    print(region)
    print("File exists:", output_path.exists())
    print("Columns:", verification_df.columns.tolist())
    print("Preview rows:", len(verification_df))

DOWNTOWN
File exists: True
Columns: ['REGION', 'TIMESTAMP', 'TOTAL_CONSUMPTION', 'Temp (°C)', 'Rel Hum (%)', 'date', 'year', 'month', 'hour', 'weekday', 'is_weekend', 'is_workday', 'is_public_holiday', 'holiday_name', 'is_daylight_saving_time', 'is_dst_transition_day']
Preview rows: 5
AIRPORT_WEST
File exists: True
Columns: ['REGION', 'TIMESTAMP', 'TOTAL_CONSUMPTION', 'Temp (°C)', 'Rel Hum (%)', 'date', 'year', 'month', 'hour', 'weekday', 'is_weekend', 'is_workday', 'is_public_holiday', 'holiday_name', 'is_daylight_saving_time', 'is_dst_transition_day']
Preview rows: 5


## Data Preparation Conclusion

The data cleaning, transformation, and integration pipeline was successfully completed for both study regions.

Six FSA-level electricity-consumption datasets were standardized and aggregated first to hourly FSA demand and subsequently into the two regional electricity-demand targets:

- **Downtown:** M5S + M5R + M6G
- **Airport-West:** L4T + M9W + M9R

Each regional demand series was integrated with its designated weather station. Temperature and relative humidity were retained as the common base meteorological variables, while additional weather variables remain available in the cleaned source data but were not propagated without an established analytical requirement.

Calendar information was integrated using the project's validated fixed UTC−5 analytical-clock convention. Temporal alignment was validated structurally and semantically, including daylight-saving-time transitions.

The resulting Downtown and Airport-West datasets each contain 43,824 unique hourly observations covering January 1, 2021 through December 31, 2025, with no missing values in electricity demand, temperature, relative humidity, or the core calendar variables.

The final datasets are therefore ready for a separate exploratory data analysis stage.

Predictive feature engineering, temporal train/test construction, peak-risk threshold estimation, and model-specific transformations are intentionally deferred to subsequent stages to preserve methodological separation and prevent data leakage.